# MuonClip experimental order parameter: Haar-connected angular susceptibility

**Method:** `haar_connected_susceptibility`

For each layer,

$$
W_t=U_t\Sigma_tV_t^{\mathsf T},
\qquad
Q_t=U_tV_t^{\mathsf T}.
$$

The hypothesis is that MuonClip strongly suppresses visible radial flow in $\Sigma_t$, while learned structure accumulates in the singular-vector geometry.  This notebook removes the instantaneous singular values, reconstructs an ordinary real matrix from that geometry, and applies standard positive-semidefinite RMT and WeightWatcher to the reconstructed Gram spectrum.

## Mathematical construction


Let

$$
C_t
=
Q_t\odot Q_t-
\mathbb E_{\mathrm{Haar}}[Q\odot Q].
$$

On the smaller coordinate side form the quartic susceptibility

$$
S_t=C_t^{\mathsf T}C_t
$$

or $S_t=C_tC_t^{\mathsf T}$ for a wide matrix.  Remove the complete finite-dimensional Haar expectation,

$$
K_t=S_t-\mathbb E_{\mathrm{Haar}}[S].
$$

Because $K_t$ is signed, reconstruct an ordinary real field with

$$
\Phi_t^{\chi}
=
\operatorname{sgn}(K_t)|K_t|^{1/2}.
$$

This is a connected four-point angular susceptibility: the universal Haar background is subtracted before spectral analysis.

## Method-matched null


The Haar expectation is estimated reproducibly for every matrix shape.  Each null replicate uses a second independent Haar/Stiefel sample passed through the identical connected construction.

## RG power-counting contract

For the transformed matrix $\Phi_t$, the implementation fixes the scalar gauge so the mean positive Gram eigenvalue is one and analyzes

$$
X_{\Phi,t}=\frac1N\Phi_t^{\mathsf T}\Phi_t.
$$

If a retained finite-window sector obeys

$$
\rho_{\Phi}(\lambda)\sim\lambda^{-\alpha_{\Phi}},
$$

then first-moment energy per logarithmic band is

$$
g_E(\lambda)=\lambda^2\rho_{\Phi}(\lambda)
\sim\lambda^{2-\alpha_{\Phi}}.
$$

Therefore

$$
y_E=2-\alpha_{\Phi},
$$

and the first-moment operator is marginal at

$$
\alpha_{\Phi}=2.
$$

This power count applies to the candidate order-parameter ESD.  It does not prove that the candidate is the correct physical quotient.  A candidate must also separate from its method-matched null, retain an acceptable finite-window fit, and become approximately stationary at late checkpoints.

## Checkpoint and WeightWatcher contract

The experiment loads the real saved step-zero checkpoint, every intermediate epoch checkpoint, and the unique best/final aliases.  It analyzes all six transformer matrices.  For every actual transformed checkpoint it calls native WeightWatcher as

```python
watcher.analyze(
    plot=True,
    savefig=str(step_dir),
    min_evals=WW_MIN_EVALS,
    randomize=True,
    ERG=False,
    fix_fingers="clip_xmax",
    max_fingers=WW_MAX_FINGERS,
)
```

The notebook displays every flow dashboard and both native ESD contact sheets inline.  There is no plot-suppression switch.

## Command-line execution

From `baseline/nanogpt_one_head`:

```bash
export RUN_DIR=/tmp/rg-nanogpt-muonclip-3ep-seed4242-20260814_090245/results/muon_clip/seed_4242
export TARGET_SEED=4242
export ANGULAR_QUOTIENT_NULLS=8
export ANGULAR_QUOTIENT_HAAR_SAMPLES=64
export MPLBACKEND=Agg

papermill \
  experiments/muonclip_angular_order_parameters/03_haar_connected_susceptibility.ipynb \
  experiments/muonclip_angular_order_parameters/03_haar_connected_susceptibility.out.ipynb \
  --log-output
```

Set `ANGULAR_QUOTIENT_FORCE=1` to recompute cached WeightWatcher results.

In [ ]:
import os

METHOD = "haar_connected_susceptibility"
EXPERIMENT_NAME = "03_haar_connected_susceptibility"
RUN_DIR = os.environ.get("RUN_DIR", "")
TARGET_SEED = int(os.environ.get("TARGET_SEED", "4242"))
WW_MIN_EVALS = int(os.environ.get("WW_MIN_EVALS", "20"))
WW_MAX_FINGERS = int(os.environ.get("WW_MAX_FINGERS", "10"))
NULL_REPLICATES = int(os.environ.get("ANGULAR_QUOTIENT_NULLS", "8"))
HAAR_SAMPLES = int(os.environ.get("ANGULAR_QUOTIENT_HAAR_SAMPLES", "64"))
DIFFUSION_MASS = float(os.environ.get("ANGULAR_QUOTIENT_DIFFUSION_MASS", "0.05"))
TEMPORAL_MAX_BLOCK = int(os.environ.get("ANGULAR_QUOTIENT_MAX_BLOCK", "8"))
FORCE = os.environ.get("ANGULAR_QUOTIENT_FORCE", "0").lower() in {"1", "true", "yes", "on"}

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown

if not RUN_DIR:
    raise EnvironmentError("Set RUN_DIR before launching this notebook")

cwd = Path.cwd().resolve()
BASELINE_ROOT = next(
    candidate for candidate in (cwd, *cwd.parents)
    if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file()
)
EXPERIMENT_DIR = BASELINE_ROOT / "experiments" / "muonclip_angular_order_parameters"
sys.path.insert(0, str(BASELINE_ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT_DIR))

from config_io import ExperimentConfig
from run_experiment import run_experiment

CONFIG = ExperimentConfig(
    run_dir=RUN_DIR,
    seed=TARGET_SEED,
    min_evals=WW_MIN_EVALS,
    max_fingers=WW_MAX_FINGERS,
    null_replicates=NULL_REPLICATES,
    haar_samples=HAAR_SAMPLES,
    diffusion_mass=DIFFUSION_MASS,
    temporal_max_block=TEMPORAL_MAX_BLOCK,
    force=FORCE,
)
print(CONFIG)

In [ ]:
RESULTS = run_experiment(
    METHOD,
    experiment_name=EXPERIMENT_NAME,
    config=CONFIG,
)
print("OUTPUT_DIR =", RESULTS["output_dir"])
display(RESULTS["checkpoint_table"])

preferred = [
    "aliases", "step", "method", "matrix_name", "status", "alpha", "y_E",
    "D", "xmin", "xmax", "num_fingers", "rand_distance",
    "null_alpha_q025", "null_alpha_median", "null_alpha_q975",
    "null_rand_q975", "alpha_outside_null", "rand_above_null",
]
endpoint = RESULTS["endpoint_summary"]
display(endpoint[[column for column in preferred if column in endpoint.columns]])

metadata = RESULTS["transform_metadata"]
display(metadata[metadata["method"] == METHOD])

## Plots

The next cell displays all RG flow dashboards, common-bin ESD trajectories, and native WeightWatcher ESD contact sheets.  The raw baseline and the candidate field are both shown.

In [ ]:
for path in RESULTS["plot_files"]:
    display(Markdown(f"### `{Path(path).name}`"))
    display(Image(filename=str(path)))

for path in RESULTS["contact_sheets"]:
    display(Markdown(f"### `{Path(path).stem}`"))
    display(Image(filename=str(path)))

## Decision rule

A positive result requires a broad accepted tail, controlled $D$, separation from the checkpoint-matched null, late stationarity, and stability under the method's explicit coarse-scale parameters.  The method is not accepted merely because one fitted value happens to be close to $\alpha=2$.